# Lab CosyVoice3 + Stage 1–2 manifest

**Model:** Fun-CosyVoice3-0.5B-2512 (docs: https://github.com/FunAudioLLM/CosyVoice)

**Ref:** đã commit trong repo tại `voice_bank/cosy_refs/` (sinh từ Kokoro, mono **16 kHz**). Clone repo là có sẵn — **không** cần upload tay.

## Thứ tự chạy

| Bước | Cell | Việc |
|------|------|------|
| 1 | Cell 1 | Config |
| 2 | Cell 2 | Clone/pull IELTS repo + prepare manifest |
| 3 | Cell 3 | **Verify ref** trong repo (wav + json) |
| 4 | Cell 4 | Clone CosyVoice + pip |
| 5 | Cell 5 | Download weight 2512 |
| 6 | Cell 6 | Smoke zero_shot 1 câu EN |
| 7 | Cell 7 | Render Part 1 từ manifest (limit 4) |
| 8 | Cell 8 | Tracking |
| 9 | Cell 9 | (Tuỳ) instruct nhanh/căng |

Không cài nặng về laptop. Không sửa `display_text`.


## Cell 1 — Config


In [1]:
# CELL 1
IELTS_REPO = "https://github.com/phamtu2x5/ielts-script2audio.git"
IELTS_DIR = "/content/ielts-script2audio"
COSY_DIR = "/content/CosyVoice"
BRANCH = "main"
print("OK Cell 1")


OK Cell 1


## Cell 2 — Clone IELTS + prepare manifest


In [2]:
# CELL 2
import os
from pathlib import Path

if not Path(IELTS_DIR).exists():
    !git clone --branch {BRANCH} {IELTS_REPO} {IELTS_DIR}
else:
    %cd {IELTS_DIR}
    !git pull origin {BRANCH}
%cd {IELTS_DIR}
!pip -q install -e ".[dev]"

Path("outputs").mkdir(exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
m = json.loads(Path("outputs/part1_manifest.json").read_text())
assert m["validation"]["valid"] is True
print("OK Cell 2 n_requests=", len(m["requests"]))


Cloning into '/content/ielts-script2audio'...
remote: Enumerating objects: 310, done.
remote: Counting objects: 100% (310/310), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 310 (delta 165), reused 231 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (310/310), 649.46 KiB | 20.29 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/ielts-script2audio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ielts-script2audio (pyproject.toml) ... done
OK Cell 2 n_requests= 15


## Cell 3 — Verify ref đã có trong repo (sau clone)


In [3]:
# CELL 3 — load committed CosyVoice refs (no manual download)
from pathlib import Path
import json
import wave

IELTS = Path(IELTS_DIR)
ref_dir = IELTS / "voice_bank" / "cosy_refs"
man_path = ref_dir / "refs_manifest.json"
map_path = ref_dir / "voice_map_runtime.json"

assert man_path.is_file(), f"Missing {man_path} — git pull main with committed refs"
assert map_path.is_file(), f"Missing {map_path}"

man = json.loads(man_path.read_text())
vmap = json.loads(map_path.read_text())

# Resolve repo-relative paths -> absolute for CosyVoice / torchaudio
for vp, entry in (vmap.get("mapping") or {}).items():
    rel = entry.get("ref_wav_relative") or entry.get("ref_wav")
    abs_path = (IELTS / rel).resolve() if not Path(rel).is_absolute() else Path(rel)
    # if relative stored as voice_bank/...
    if not abs_path.is_file():
        abs_path = (IELTS / "voice_bank" / "cosy_refs" / Path(rel).name).resolve()
    entry["ref_wav"] = str(abs_path)
    entry["ref_wav_relative"] = str(Path("voice_bank/cosy_refs") / abs_path.name)
    with wave.open(str(abs_path), "rb") as w:
        ch, sw, sr, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
    sec = round(n / sr, 2)
    print(vp, abs_path.name, f"ch={ch} sw={sw} sr={sr} sec={sec}")
    assert ch == 1 and sw == 2 and sr == 16000, "ref must be mono int16 16kHz"
    assert sec >= 5.0, f"ref too short: {sec}s"
    assert abs_path.is_file()

# CosyVoice model dir lives under CosyVoice clone (set after Cell 5 download)
vmap["model_dir"] = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")
runtime_map = ref_dir / "voice_map_runtime.colab.json"
runtime_map.write_text(json.dumps(vmap, ensure_ascii=False, indent=2) + "\n")
print("OK Cell 3 refs ready ->", runtime_map)


vp_spk_a spk_a_ref.wav ch=1 sw=2 sr=16000 sec=8.4
vp_spk_b spk_b_ref.wav ch=1 sw=2 sr=16000 sec=10.03
OK Cell 3 refs ready -> /content/ielts-script2audio/voice_bank/cosy_refs/voice_map_runtime.colab.json


## Cell 4 — Clone CosyVoice + requirements (docs official)


In [4]:
# CELL 4 — CosyVoice install (Colab)
# requirements.txt may partially fail on Colab; we install critical imports explicitly.
import os
import sys
import subprocess
from pathlib import Path

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
%cd {COSY_DIR}
!git submodule update --init --recursive

# Best-effort full requirements (may error on some wheels — do not stop)
import subprocess as _sp
print("pip install -r requirements.txt (best effort)...")
_rc = _sp.call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("requirements.txt exit code:", _rc)

# Critical packages for: from cosyvoice.cli.cosyvoice import AutoModel
# - HyperPyYAML  (import hyperpyyaml)
# - onnxruntime  (CPU; avoid onnxruntime-gpu → libcudart.so.13 on Colab)
# - openai-whisper (import whisper)  ← current Colab error
# - modelscope, torchaudio, inflect, etc. often needed by frontend
!pip -q uninstall -y onnxruntime-gpu 2>/dev/null || true
pipi(
    "HyperPyYAML",
    "modelscope",
    "torchaudio",
    "onnxruntime",
    "openai-whisper",
    "inflect",
    "wetext",
    "gdown",
)

!apt-get -qq update && apt-get -qq install -y sox libsox-dev ffmpeg || true

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

import onnxruntime
print("onnxruntime", getattr(onnxruntime, "__version__", "?"))
from hyperpyyaml import load_hyperpyyaml
import whisper
print("whisper", getattr(whisper, "__version__", "ok"))
from cosyvoice.cli.cosyvoice import AutoModel
print("OK Cell 4 import AutoModel")
print("OK Cell 4", os.getcwd())


## Cell 5 — Download Fun-CosyVoice3-0.5B-2512


In [ ]:
# CELL 5 — weights (slow first time)
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B"
model_dir.parent.mkdir(parents=True, exist_ok=True)
snapshot_download(
    "FunAudioLLM/Fun-CosyVoice3-0.5B-2512",
    local_dir=str(model_dir),
)
print("OK Cell 5", model_dir, "exists=", model_dir.exists())


## Cell 6 — Smoke zero_shot EN (docs CosyVoice3)


In [ ]:
# CELL 4 — CosyVoice install (Colab)
# requirements.txt may partially fail on Colab; we install critical imports explicitly.
import os
import sys
import subprocess
from pathlib import Path

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
%cd {COSY_DIR}
!git submodule update --init --recursive

# Best-effort full requirements (may error on some wheels — do not stop)
import subprocess as _sp
print("pip install -r requirements.txt (best effort)...")
_rc = _sp.call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("requirements.txt exit code:", _rc)

# Critical packages for: from cosyvoice.cli.cosyvoice import AutoModel
# - HyperPyYAML  (import hyperpyyaml)
# - onnxruntime  (CPU; avoid onnxruntime-gpu → libcudart.so.13 on Colab)
# - openai-whisper (import whisper)  ← current Colab error
# - modelscope, torchaudio, inflect, etc. often needed by frontend
!pip -q uninstall -y onnxruntime-gpu 2>/dev/null || true
pipi(
    "HyperPyYAML",
    "modelscope",
    "torchaudio",
    "onnxruntime",
    "openai-whisper",
    "inflect",
    "wetext",
    "gdown",
)

!apt-get -qq update && apt-get -qq install -y sox libsox-dev ffmpeg || true

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

import onnxruntime
print("onnxruntime", getattr(onnxruntime, "__version__", "?"))
from hyperpyyaml import load_hyperpyyaml
import whisper
print("whisper", getattr(whisper, "__version__", "ok"))
from cosyvoice.cli.cosyvoice import AutoModel
print("OK Cell 4 import AutoModel")
print("OK Cell 4", os.getcwd())


## Cell 7 — Render Part 1 manifest (limit 4)


In [ ]:
# CELL 4 — CosyVoice install (Colab)
# requirements.txt may partially fail on Colab; we install critical imports explicitly.
import os
import sys
import subprocess
from pathlib import Path

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
%cd {COSY_DIR}
!git submodule update --init --recursive

# Best-effort full requirements (may error on some wheels — do not stop)
import subprocess as _sp
print("pip install -r requirements.txt (best effort)...")
_rc = _sp.call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("requirements.txt exit code:", _rc)

# Critical packages for: from cosyvoice.cli.cosyvoice import AutoModel
# - HyperPyYAML  (import hyperpyyaml)
# - onnxruntime  (CPU; avoid onnxruntime-gpu → libcudart.so.13 on Colab)
# - openai-whisper (import whisper)  ← current Colab error
# - modelscope, torchaudio, inflect, etc. often needed by frontend
!pip -q uninstall -y onnxruntime-gpu 2>/dev/null || true
pipi(
    "HyperPyYAML",
    "modelscope",
    "torchaudio",
    "onnxruntime",
    "openai-whisper",
    "inflect",
    "wetext",
    "gdown",
)

!apt-get -qq update && apt-get -qq install -y sox libsox-dev ffmpeg || true

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

import onnxruntime
print("onnxruntime", getattr(onnxruntime, "__version__", "?"))
from hyperpyyaml import load_hyperpyyaml
import whisper
print("whisper", getattr(whisper, "__version__", "ok"))
from cosyvoice.cli.cosyvoice import AutoModel
print("OK Cell 4 import AutoModel")
print("OK Cell 4", os.getcwd())


## Cell 8 — Tracking


In [ ]:
# CELL 8 — track
from pathlib import Path
from IPython.display import Audio, display
import json

rep = Path(IELTS_DIR) / "lab_audio/cosyvoice3_part1/lab_render_report.json"
assert rep.is_file(), "Chay Cell 7 truoc"
report = json.loads(rep.read_text())
audio_dir = rep.parent
for i, seg in enumerate(report.get("segments") or [], 1):
    print("=" * 72)
    print(f"[{i}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('status')}")
    print("DISPLAY:", seg.get("display_text"))
    print("SPOKEN :", seg.get("spoken_text"))
    print("REF    :", seg.get("ref_wav"))
    fn = seg.get("output_filename")
    wav = audio_dir / fn if fn else None
    if wav and wav.is_file():
        display(Audio(filename=str(wav)))
    elif seg.get("error"):
        print("ERROR", seg["error"])
print("OK Cell 8")


## Cell 9 — (Tuỳ) instruct nhanh/căng


In [ ]:
# CELL 4 — CosyVoice install (Colab)
# requirements.txt may partially fail on Colab; we install critical imports explicitly.
import os
import sys
import subprocess
from pathlib import Path

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
%cd {COSY_DIR}
!git submodule update --init --recursive

# Best-effort full requirements (may error on some wheels — do not stop)
import subprocess as _sp
print("pip install -r requirements.txt (best effort)...")
_rc = _sp.call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("requirements.txt exit code:", _rc)

# Critical packages for: from cosyvoice.cli.cosyvoice import AutoModel
# - HyperPyYAML  (import hyperpyyaml)
# - onnxruntime  (CPU; avoid onnxruntime-gpu → libcudart.so.13 on Colab)
# - openai-whisper (import whisper)  ← current Colab error
# - modelscope, torchaudio, inflect, etc. often needed by frontend
!pip -q uninstall -y onnxruntime-gpu 2>/dev/null || true
pipi(
    "HyperPyYAML",
    "modelscope",
    "torchaudio",
    "onnxruntime",
    "openai-whisper",
    "inflect",
    "wetext",
    "gdown",
)

!apt-get -qq update && apt-get -qq install -y sox libsox-dev ffmpeg || true

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

import onnxruntime
print("onnxruntime", getattr(onnxruntime, "__version__", "?"))
from hyperpyyaml import load_hyperpyyaml
import whisper
print("whisper", getattr(whisper, "__version__", "ok"))
from cosyvoice.cli.cosyvoice import AutoModel
print("OK Cell 4 import AutoModel")
print("OK Cell 4", os.getcwd())


Xong lab. not_selected. Ref Kokoro synthetic committed for Colab clone.
